# diffusion models: ddpm noising / denoising basics

working through the ho et al 2020 ddpm paper at a slow pace. this notebook only covers the forward (noising) and a hand-rolled reverse step on toy 2d data, no unet yet.

In [ ]:
import torch
import math

T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
print(alpha_bars[0].item(), alpha_bars[-1].item())


In [ ]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[t].view(-1, 1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise, noise

# 2-d swiss roll-ish
theta = torch.rand(2000) * 4 * math.pi
x0 = torch.stack([theta * torch.cos(theta), theta * torch.sin(theta)], dim=1) / 10.0

t = torch.randint(0, T, (x0.size(0),))
xt, noise = q_sample(x0, t)
print(x0.shape, xt.shape)


## tiny mlp eps-predictor

just to confirm the loss flows.

In [ ]:
import torch.nn as nn

class EpsNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x, t):
        t_norm = (t.float().unsqueeze(1)) / T
        return self.net(torch.cat([x, t_norm], dim=1))

model = EpsNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(500):
    idx = torch.randint(0, x0.size(0), (256,))
    x_b = x0[idx]
    t_b = torch.randint(0, T, (256,))
    xt_b, eps_b = q_sample(x_b, t_b)
    eps_hat = model(xt_b, t_b)
    loss = ((eps_hat - eps_b) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(step, loss.item())


tomorrow: hook up the reverse sampling loop and plot trajectories. unet later.

In [ ]:
# bump lora rank
R = 16